# Whisper-small + LoRA on PriMock57 (Kaggle / Colab, free T4)

Fine-tunes `openai/whisper-small` with LoRA on the **train** consultations of PriMock57
(CC BY 4.0) and reports WER/CER on the **held-out test consultations**, before and after,
with consultation-level bootstrap CIs and a paired comparison.

Settings: *Accelerator = GPU T4*, *Internet = on*. Runtime is roughly 1 h end to end on a T4
(baseline eval, about 3 epochs of training, fine-tuned eval).

Everything runs through the same CLI as the laptop runs, so numbers are comparable.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/vik-bit/medscribe.git
%cd medscribe
!pip install -q -e ".[asr]"

In [ ]:
# ~1.7 GB of audio via git-lfs, pinned commit
!bash scripts/fetch_primock57.sh
!python -m medscribe splits

## 1. Baseline (zero-shot whisper-small) on the full test split

In [ ]:
!python -m medscribe asr-eval --split test --batch-size 16     --out results/asr_test_base.json --preds results/asr_test_base.jsonl

## 2. LoRA fine-tuning (plain PyTorch loop, fp16 autocast)
Best adapter is selected on the **dev** split, never on test.

In [ ]:
!python -m medscribe asr-train --epochs 3 --batch-size 16 --lr 1e-3     --lora-r 32 --lora-alpha 64 --eval-every 200 --out checkpoints/whisper-small-lora

## 3. Fine-tuned model on the same test utterances + paired comparison

In [ ]:
!python -m medscribe asr-eval --split test --batch-size 16     --adapter checkpoints/whisper-small-lora/best     --out results/asr_test_lora.json --preds results/asr_test_lora.jsonl
!python -m medscribe asr-compare --a results/asr_test_lora.jsonl --b results/asr_test_base.jsonl     --out results/asr_test_compare.json

In [ ]:
# Download these and commit them to results/ (the README only reports numbers from these files)
!cp checkpoints/whisper-small-lora/train_log.json results/asr_train_log.json
!zip -qr /kaggle/working/medscribe_asr_results.zip results checkpoints/whisper-small-lora/best